In [5]:
from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [1]:
from pathlib import Path

data_path = Path("../data/physionet.org/files/eegmmidb/1.0.0")

edf_files = [edf_file for edf_file in data_path.rglob("*.edf")]
len(edf_files)

1526

In [7]:
import mne

from src.eeg.preprocessing import prepare_motor_imagery_epochs

selected_runs = ["R04", "R08", "R12"]
selected_channels = ["C3", "Cz", "C4"]

for edf_file in edf_files:
    subj_id = edf_file.parent.name
    run_id = edf_file.stem[-3:]

    if run_id not in selected_runs:
        continue

    try:
        raw = mne.io.read_raw_edf(edf_file, verbose=False)
        
        epochs, labels, metadata = prepare_motor_imagery_epochs(raw)
        
    except Exception as e:
        print(e)
        continue
        

Reading 0 ... 19679  =      0.000 ...   122.994 secs...


Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19839  =      0.000 ...   123.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19839  =      0.000 ...   123.994 secs...
Reading 0 ... 19839  =      0.000 ...   123.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 

/tmp/ipykernel_4056087/576421725.py:16: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_file, verbose=False)
/tmp/ipykernel_4056087/576421725.py:16: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_file, verbose=False)
/tmp/ipykernel_4056087/576421725.py:16: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_file, verbose=False)


Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Reading 0 ... 19679  =      0.000 ...   122.994 

In [9]:
bands = {
    "mu": (8.0, 13.0),
    "beta": (13.0, 30.0),
}

psd = epochs.compute_psd(
    method="welch",
    fmin=min(low for low, _ in bands.values()),
    fmax=max(high for _, high in bands.values()),
    verbose=False,
)
psd

<Power Spectrum (from Epochs, welch method) | 15 epochs × 3 channels × 66 freqs, 8.3-29.9 Hz>

In [12]:
import numpy as np
import pandas as pd

epsilon = 1e-12

powers = psd.get_data()
freqs = psd.freqs

feature_columns = []
feature_names = []

for channel_index, channel_name in enumerate(epochs.ch_names):
    for band_name, (low_freq, high_freq) in bands.items():
        band_mask = (freqs >= low_freq) & (freqs < high_freq)
        if not np.any(band_mask):
            band_power = np.full(len(epochs), epsilon)
        else:
            band_power = powers[:, channel_index, :][:, band_mask].mean(axis=1)
        feature_columns.append(np.log(band_power + epsilon))
        feature_names.append(f"{channel_name}_{band_name}")

X = np.column_stack(feature_columns)
features_df = pd.DataFrame(X, columns=feature_names)

In [13]:
features_df

,C3.._mu,C3.._beta,Cz.._mu,Cz.._beta,C4.._mu,C4.._beta
0,-25.442233,-25.521487,-25.465747,-25.587767,-25.795915,-26.053146
1,-24.826595,-25.674871,-24.605756,-25.756215,-24.883382,-26.077300
2,-24.444715,-25.638075,-24.395394,-25.752990,-25.398916,-26.048041
3,-25.402831,-25.762921,-25.005639,-25.815862,-25.238237,-26.031896
4,-24.974162,-25.831584,-25.154846,-25.875638,-25.287337,-26.166780
5,-24.912559,-25.811510,-24.925696,-25.793274,-25.413619,-26.085178
6,-24.671439,-25.557581,-25.262734,-25.765926,-25.862431,-26.279963
7,-24.628600,-25.559211,-24.747866,-25.724992,-25.192397,-26.083403
8,-25.313032,-25.885997,-25.234683,-26.165622,-25.702194,-26.518487
9,-24.715811,-25.674962,-24.787553,-25.579929,-25.101177,-25.934581
